# 02 — Preprocessing and Train/Val/Test Split

This notebook turns the raw GermEval 2018 files into clean `train.csv` / `val.csv` / `test.csv` files under `data/processed/`. It builds on the findings from `dataset-exploration.ipynb` plus a few additional checks on the raw files noted below.

All reusable logic lives in `src/data.py` and `src/config.py`.

**Official evaluation metric: macro-F1** (not accuracy), because of the class imbalance (~66% OTHER / ~34% OFFENSE).

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, data

## 1. Load raw data

The GermEval 2018 shared task ships two files that already form an official train/test split:

- `germeval2018.training.txt`
- `germeval2018.test.txt`

Using the official split 

In [2]:
train_raw, test_raw = data.load_germeval(
    config.RAW_DIR,
    train_filename=config.TRAIN_FILENAME,
    test_filename=config.TEST_FILENAME,
)

print("train_raw:", train_raw.shape)
print("test_raw: ", test_raw.shape)
print()
print("Raw label counts (train):")
print(train_raw["label"].value_counts())
print()
print("Raw label counts (test):")
print(test_raw["label"].value_counts())

train_raw: (5009, 3)
test_raw:  (3532, 3)

Raw label counts (train):
label
OTHER      3321
OFFENSE    1688
Name: count, dtype: int64

Raw label counts (test):
label
OTHER      2330
OFFENSE    1202
Name: count, dtype: int64


## 2. Text cleaning

Cleaning is intentionally minimal and limited to artifacts that are actually present in this dataset:

1. **`|LBR|` removal** — a literal token used in the original export to mark line breaks inside a tweet (1287 occurrences in training, 675 in test). It is not real content, so it is replaced with a space.
2. **Mention masking** — `@handle` is replaced with a fixed `@USER` placeholder. About 68% of tweets contain a mention; masking avoids the model keying on specific high-cardinality usernames and avoids storing real handles in the processed data.
3. **Whitespace normalization** — repeated whitespace (partly introduced by step 1) is collapsed, and the text is stripped.

**Deliberately left untouched:** casing, punctuation, hashtags, and emojis. There is no evidence they are noise and all of them may carry information relevant to offensive-language detection.

Both the raw (`text_raw`) and cleaned (`text`) columns are kept in the saved data, so later notebooks (e.g. the perturbation experiments) can still access the original if needed.

In [3]:
train_clean = data.preprocess_dataframe(train_raw)
test_clean = data.preprocess_dataframe(test_raw)

print("train_clean:", train_clean.shape, "(from", train_raw.shape[0], "raw rows)")
print("test_clean: ", test_clean.shape, "(from", test_raw.shape[0], "raw rows)")

# before/after example on a row that contains the |LBR| artifact
example_raw = train_raw.loc[train_raw["text_raw"].str.contains(r"\|LBR\|", regex=True), "text_raw"].iloc[0]
print()
print("before:", example_raw)
print("after: ", data.clean_text(example_raw))

train_clean: (5009, 4) (from 5009 raw rows)
test_clean:  (3532, 4) (from 3532 raw rows)

before: @Ralf_Stegner Oman Ralle..dich mag ja immer noch keiner. Du willst das die Hetze gegen dich aufhört? |LBR| Geh in Rente und verzichte auf die 1/2deiner Pension
after:  @USER Oman Ralle..dich mag ja immer noch keiner. Du willst das die Hetze gegen dich aufhört? Geh in Rente und verzichte auf die 1/2deiner Pension


## 3. Integrity check after cleaning

`preprocess_dataframe` drops exact-duplicate and empty texts defensively. `01_dataset_exploration.ipynb` already found zero duplicate comments across the combined data, so it is expected for zero rows dropped here.

In [4]:
print("Rows dropped in train (duplicates/empty):", train_raw.shape[0] - train_clean.shape[0])
print("Rows dropped in test (duplicates/empty): ", test_raw.shape[0] - test_clean.shape[0])
print()
print("Any nulls in train_clean:", train_clean.isnull().any().any())
print("Any nulls in test_clean: ", test_clean.isnull().any().any())

Rows dropped in train (duplicates/empty): 0
Rows dropped in test (duplicates/empty):  0

Any nulls in train_clean: False
Any nulls in test_clean:  False


## 4. Train/validation split

The official test file (`test_clean`) is held out untouched from here on, it is only loaded and cleaned with the same deterministic function, never explored further, and will not be looked at again until the final evaluation of the whole project.

A validation set is carved out of the training pool (`train_clean`) using `sklearn.model_selection.train_test_split`:

I split the training data into train and validation sets using stratification on the binary `label` (OFFENSE/OTHER). Since this is the target of the model, it makes sense to preserve its class distribution in both splits.

I did not stratify on `label_fine` because some of its classes are quite small, which could result in very few examples in the validation set.

The validation set contains **15%** of the training data, and I fixed the random seed to **42** so that anyone can reproduce the exact same split.

In [5]:
train_split, val_split = data.make_splits(
    train_clean,
    val_size=config.VAL_SIZE,
    seed=config.SEED,
    stratify_col="label",
)

print("train_split:", train_split.shape)
print("val_split:  ", val_split.shape)
print("test_clean: ", test_clean.shape)

train_split: (4257, 4)
val_split:   (752, 4)
test_clean:  (3532, 4)


## 5. Sanity Checks

Before saving the processed datasets, I checked two things:

- The class distribution stays almost the same across the train, validation, and test sets. Since the split is stratified on the binary label, the OTHER/OFFENSE ratio should remain very similar in each split.
- There is no data leakage. Duplicate texts within a split are removed, and the same text is never present in more than one split.

In [6]:
splits = {"train": train_split, "val": val_split, "test": test_clean}

counts = pd.DataFrame({name: df["label"].value_counts() for name, df in splits.items()})
proportions = pd.DataFrame({name: df["label"].value_counts(normalize=True).round(3) for name, df in splits.items()})

print("Absolute counts:")
display(counts)
print()
print("Proportions:")
display(proportions)

Absolute counts:


,train,val,test
label,,,
OTHER,2822,499,2330
OFFENSE,1435,253,1202



Proportions:


,train,val,test
label,,,
OTHER,0.663,0.664,0.66
OFFENSE,0.337,0.336,0.34


In [7]:
train_texts = set(train_split["text"])
val_texts = set(val_split["text"])
test_texts = set(test_clean["text"])

print("Duplicate texts within train:", train_split["text"].duplicated().sum())
print("Duplicate texts within val:  ", val_split["text"].duplicated().sum())
print("Duplicate texts within test: ", test_clean["text"].duplicated().sum())
print()
print("Overlap train/val:  ", len(train_texts & val_texts))
print("Overlap train/test: ", len(train_texts & test_texts))
print("Overlap val/test:   ", len(val_texts & test_texts))

Duplicate texts within train: 0
Duplicate texts within val:   0
Duplicate texts within test:  0

Overlap train/val:   0
Overlap train/test:  0
Overlap val/test:    0


## 6. Save processed splits

Saved columns: `text_raw`, `text`, `label`, `label_fine`. The cleaned `text` is what modeling notebooks should use; `text_raw` and `label_fine` are kept for reference and error analysis.

In [8]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_columns = ["text_raw", "text", "label", "label_fine"]

train_split[output_columns].to_csv(config.PROCESSED_DIR / "train.csv", index=False)
val_split[output_columns].to_csv(config.PROCESSED_DIR / "val.csv", index=False)
test_clean[output_columns].to_csv(config.PROCESSED_DIR / "test.csv", index=False)

print("Saved:")
for name, df in [("train.csv", train_split), ("val.csv", val_split), ("test.csv", test_clean)]:
    print(f"  {name}: {df.shape[0]} rows")

Saved:
  train.csv: 4257 rows
  val.csv: 752 rows
  test.csv: 3532 rows
